# AutoEIT — Whisper-large-v2 LoRA Fine-tuning (Kaggle GPU)

### Changes from previous version
| Area | Before | After | Reason |
|---|---|---|---|
| **Base model** | whisper-small (74 M) | whisper-large-v2 (1.55 B) | 20× capacity gap vs baseline large-v3 |
| **LoRA targets** | encoder-only + duplicates | all attn projections incl. decoder cross-attn | cross-attn is where L2 acoustic→text mapping happens |
| **Training budget** | 1500 steps (<1 epoch) | 10 epochs, cosine LR | was severely undertrained; best ckpt was step 600 |
| **Augmentation** | pre-built files only | + in-flight SpecAugment | `augment` flag was stored but never used |
| **Hallucinations** | none | repetition penalty + post-processor | "ehm ehm ehm…×40" seen in predictions |
| **Eval coverage** | 200/4033 dev samples | 1000/4033 dev samples | noisy checkpoint selection |

### Setup checklist before running
1. **Runtime** → Accelerator → **GPU T4 x2** (or P100/A100). Enable internet.
2. On your Mac, rebuild the mel cache then zip the `data/` directory:
   ```bash
   cd /path/to/autoeit
   source .venv/bin/activate
   rm -rf data/mel_cache/
   python -c "from src.data_loader import MelCache; MelCache().precompute_all()"
   zip -r eit-data.zip data/ \
       --exclude "data/transcripts/backups/*" \
       --exclude "*.pyc" --exclude "__pycache__/*"
   ```
   Upload the resulting `eit-data.zip` as a **new Kaggle Dataset** named **`eit-data`**.

> **Expected training time:** ~8–10 h on T4 × 2  ·  ~4–5 h on A100
>
> **Memory:** whisper-large-v2 fp32 ≈ 6.2 GB per GPU.  With LoRA r=8, gradient
> checkpointing, and batch 4, peak VRAM is ~9–10 GB — comfortable on a 15 GB T4.


## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    accelerate>=0.29.0 \
    datasets>=2.18.0 \
    jiwer>=3.0.3 \
    soundfile \
    librosa \
    evaluate \
    bitsandbytes \
    tensorboard

## Cell 2 — Imports

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import math
import os
import random
import re
import sys
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import jiwer
import librosa
import numpy as np
import pandas as pd

try:
    import soundfile as sf
    _HAS_SOUNDFILE = True
except ImportError:
    _HAS_SOUNDFILE = False

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("autoeit.kaggle")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s  |  GPU: %s",
            DEVICE,
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Cell 3 — Configuration  ← **edit this cell**

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

_KAGGLE_DATA_ROOT = "/kaggle/input/datasets/tilak11612/eit-data/data"

CFG = dict(
    # ── Paths ───────────────────────────────────────────────────────────────
    data_root          = _KAGGLE_DATA_ROOT,
    audio_root         = _KAGGLE_DATA_ROOT,
    train_csv          = f"{_KAGGLE_DATA_ROOT}/transcripts/train_augmented.csv",
    dev_csv            = f"{_KAGGLE_DATA_ROOT}/transcripts/dev.csv",
    test_csv           = f"{_KAGGLE_DATA_ROOT}/transcripts/test.csv",
    manifest_csv       = f"{_KAGGLE_DATA_ROOT}/transcripts/processed_manifest.csv",
    mel_cache_dir      = f"{_KAGGLE_DATA_ROOT}/mel_cache",
    mel_cache_writable = "/kaggle/working/mel_cache",
    output_dir         = "/kaggle/working/checkpoints",
    results_dir        = "/kaggle/working/results",
    log_dir            = "/kaggle/working/logs",

    # ── Model ───────────────────────────────────────────────────────────────
    # FIX 1: Upgraded from whisper-small (74 M) → whisper-large-v2 (1.55 B).
    # The small model had a 20× capacity gap vs the large-v3 baseline and could
    # not represent the acoustic diversity of L2 learner speech.
    base_model         = "openai/whisper-large-v2",
    language           = "es",
    task               = "transcribe",

    # ── LoRA ────────────────────────────────────────────────────────────────
    # FIX 2: Clean target_modules list — ["q_proj","v_proj","k_proj","out_proj"]
    # PEFT matches these short names against every module in the model, covering:
    #   • encoder self-attention
    #   • decoder self-attention
    #   • decoder CROSS-attention  ← was missing before (most critical for L2)
    # The previous list had redundant wildcard patterns AND missed cross-attn.
    # r=8 on large-v2 ≈ 10.5 M trainable params; r=16 on small ≈ 4 M — so the
    # total adapter footprint is comparable while covering far more of the model.
    lora_r             = 8,
    lora_alpha         = 16,
    lora_dropout       = 0.05,
    target_modules     = ["q_proj", "v_proj", "k_proj", "out_proj"],

    # ── Training ────────────────────────────────────────────────────────────
    # FIX 3: Epoch-based training with cosine LR decay.
    # Before: max_steps=1500 → <1 epoch on 56 K samples; best ckpt at step 600
    # means the model never stabilised.  10 epochs with cosine decay allows
    # thorough convergence and the LR anneals naturally to near-zero.
    #
    # Batch reduced 8→4 to fit large-v2 on a 15 GB T4.
    # grad_accum raised 4→8 so effective batch stays at 32 (unchanged).
    batch_size         = 4,
    grad_accum         = 8,     # effective batch = 4 × 8 = 32
    learning_rate      = 5e-6,  # lower than before; large models need gentler LR
    warmup_ratio       = 0.05,  # 5% of total steps → ~875 warmup steps @ 10 epochs
    num_train_epochs   = 10,
    lr_scheduler_type  = "cosine",
    eval_steps         = 500,   # also used as save_steps
    save_steps         = 500,
    logging_steps      = 25,
    weight_decay       = 1e-2,
    fp16               = True,
    dataloader_workers = 4,
    early_stopping_patience = 3,   # patience in eval-steps units

    # ── Eval / generation ───────────────────────────────────────────────────
    # FIX 4 (partial): Increased eval coverage 200→1000 for better ckpt selection.
    # FIX 5: repetition_penalty + no_repeat_ngram_size guard against the
    # "ehm ehm ehm…×40" hallucination failure mode seen on confusing L2 inputs.
    generation_max_length  = 80,
    max_eval_gen_samples   = 1000,
    repetition_penalty     = 1.3,
    no_repeat_ngram_size   = 4,

    # ── SpecAugment (FIX 4) ─────────────────────────────────────────────────
    # Applied in-flight inside EITDataset._get_mel() when augment=True.
    # The augment flag was stored in the old code but never actually used.
    # Standard Whisper SpecAugment parameters from the original paper:
    #   • 2 time masks, each up to 100 frames  (out of 3000 total)
    #   • 2 frequency masks, each up to 20 bins (out of 80 total)
    specaugment_time_masks      = 2,
    specaugment_time_mask_width = 100,
    specaugment_freq_masks      = 2,
    specaugment_freq_mask_width = 20,

    # ── Data constants ──────────────────────────────────────────────────────
    target_sr          = 16_000,
    max_audio_samples  = 480_000,   # 30 s
    max_label_len      = 224,

    # ── Dataset sampling weights ────────────────────────────────────────────
    dataset_weights    = {
        "Nebrija-INMIGRA": 0.35,
        "Nebrija-WOCAE":   0.20,
        "SPLLOC1":         0.45,
    },
)

for d in [CFG["mel_cache_writable"], CFG["output_dir"],
          CFG["results_dir"], CFG["log_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

logger.info("Config loaded.")
logger.info("  Base model : %s", CFG["base_model"])
logger.info("  LoRA targets: %s", CFG["target_modules"])
logger.info("  Epochs     : %d  |  LR: %g  |  Scheduler: %s",
            CFG["num_train_epochs"], CFG["learning_rate"], CFG["lr_scheduler_type"])
logger.info("  Eff. batch : %d  (per_device=%d × accum=%d)",
            CFG["batch_size"] * CFG["grad_accum"],
            CFG["batch_size"], CFG["grad_accum"])

## Cell 4 — Text normalisation, metrics & post-processing

In [ ]:
_PUNCT_RE = re.compile(r"[^\w\s']", re.UNICODE)
_MULTI_WS  = re.compile(r"\s{2,}")

def normalize_text(text: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    if not text or not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<unintelligible>", "", text, flags=re.IGNORECASE)
    text = _PUNCT_RE.sub("", text)
    text = _MULTI_WS.sub(" ", text)
    return text.strip()


# FIX 5 — post-processing hallucination guard ─────────────────────────────────
# Whisper hallucinates on difficult L2 inputs by looping on filler tokens.
# This function truncates the transcript at the first repetition loop and
# collapses responses that are nothing but disfluency markers.

_FILLER_TOKENS = {"eh", "ehm", "um", "uh", "mm", "hmm", "ah", "oh",
                  "hem", "hm", "er", "m"}

def post_process_transcript(text: str, max_consecutive_repeat: int = 4) -> str:
    """
    Remove hallucination artefacts from a decoded transcript.

    1.  Truncates at the first run where the same token repeats more than
        `max_consecutive_repeat` times consecutively.
    2.  Returns empty string when the whole output consists only of
        disfluency/filler tokens (e.g. 'eh', 'ehm', 'um').

    Parameters
    ----------
    text : str
        Raw decoded text (already lowercase, already normalised).
    max_consecutive_repeat : int
        Maximum allowed consecutive identical tokens before truncation.
    """
    if not text or not text.strip():
        return ""

    tokens = text.strip().split()
    if not tokens:
        return ""

    # Truncate at first runaway repetition loop
    result: List[str] = [tokens[0]]
    run = 1
    for tok in tokens[1:]:
        if tok == result[-1]:
            run += 1
            if run <= max_consecutive_repeat:
                result.append(tok)
            else:
                # Everything from here is very likely a hallucination loop
                break
        else:
            result.append(tok)
            run = 1

    # Collapse filler-only outputs to empty string
    if all(t.lower() in _FILLER_TOKENS for t in result):
        return ""

    return " ".join(result)


def compute_metrics_raw(
    references: Sequence[str],
    hypotheses: Sequence[str],
) -> Dict[str, float]:
    refs = [r if r.strip() else "<empty>" for r in references]
    hyps = [h if h.strip() else "<empty>" for h in hypotheses]
    wer_val = float(jiwer.wer(refs, hyps))
    cer_val = float(jiwer.cer(refs, hyps))
    agree   = sum(1 for r, h in zip(refs, hyps)
                  if (1.0 - jiwer.wer(r, h)) >= 0.90)
    return {
        "wer":             wer_val,
        "cer":             cer_val,
        "90pct_agreement": agree / max(len(refs), 1),
    }


logger.info("normalize_text, post_process_transcript, compute_metrics_raw ready.")

## Cell 5 — EITDataset  (mel cache + tokenisation + SpecAugment)

In [ ]:
class EITDataset(Dataset):
    """
    Lazy-loading EIT dataset.

    Changes from previous version
    ─────────────────────────────
    FIX 4 — SpecAugment is now actually applied when augment=True.
    The previous code stored self.augment but never used it.  The new
    _apply_specaugment() method applies time-masking and frequency-masking
    directly on the pre-computed mel spectrogram before it is returned to
    the collator.  This runs on CPU inside the DataLoader worker, adding
    negligible overhead while meaningfully improving robustness.

    Path remapping
    ──────────────
    CSVs store processed_path as relative paths:
        data/processed/Nebrija-INMIGRA/foo.wav
    audio_root tells the dataset where data/ lives so that:
        data/X/Y  →  {audio_root}/X/Y
    Mel-cache keys use the original relative path so hashes match
    pre-built .npy files uploaded from your Mac.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        processor: WhisperProcessor,
        mel_cache_dir: str = "/kaggle/working/mel_cache",
        audio_root: str    = "/kaggle/input/eit-data",
        max_audio_len: int = 480_000,
        max_label_len: int = 224,
        augment: bool      = False,
    ) -> None:
        self.df            = df.reset_index(drop=True)
        self.processor     = processor
        self.fe            = processor.feature_extractor
        self.tok           = processor.tokenizer
        self.max_audio_len = max_audio_len
        self.max_label_len = max_label_len
        self.augment       = augment
        self._audio_root   = audio_root.rstrip("/")

        # Special token ids
        self._sot      = self.tok.convert_tokens_to_ids("<|startoftranscript|>")
        self._eot      = self.tok.eos_token_id
        self._pad      = self._eot
        forced         = processor.get_decoder_prompt_ids(language="es",
                                                          task="transcribe")
        self._lang_tok    = forced[0][1]
        self._transcribe  = forced[1][1] if len(forced) > 1 else self._sot
        self._no_ts       = self.tok.convert_tokens_to_ids("<|notimestamps|>")
        self._prefix      = [self._sot, self._lang_tok,
                              self._transcribe, self._no_ts]

        self._mel_dir  = Path(mel_cache_dir)
        self._mel_dir.mkdir(parents=True, exist_ok=True)
        self._tok_cache: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row            = self.df.iloc[idx]
        mel            = self._get_mel(idx, row)          # (3000, 80) float32
        dec_in, labels = self._get_tokens(idx, row)       # (T,) int64 each
        return {
            "input_features":    torch.tensor(mel, dtype=torch.float32).T,  # (80,3000)
            "decoder_input_ids": torch.tensor(dec_in, dtype=torch.long),
            "labels":            torch.tensor(labels, dtype=torch.long),
        }

    # ── Path remapping ───────────────────────────────────────────────────────

    def _remap_path(self, path: str) -> str:
        if os.path.isabs(path):
            return path
        if path.startswith("data/"):
            path = path[len("data/"):]
        return os.path.join(self._audio_root, path)

    # ── Mel ──────────────────────────────────────────────────────────────────

    def _cache_key(self, row: pd.Series) -> str:
        return hashlib.md5(str(row.processed_path).encode()).hexdigest()

    @staticmethod
    def _safe_load_npy(path: Path) -> Optional[np.ndarray]:
        try:
            return np.load(path)
        except (EOFError, ValueError, OSError):
            try:
                path.unlink(missing_ok=True)
            except OSError:
                pass
            return None

    def _get_mel(self, idx: int, row: pd.Series) -> np.ndarray:
        # 1) Try read-only Kaggle input cache (uploaded from local Mac)
        ro_file = Path(CFG["mel_cache_dir"]) / f"{self._cache_key(row)}.npy"
        if ro_file.exists():
            mel = self._safe_load_npy(ro_file)
            if mel is not None:
                # FIX 4 — apply SpecAugment in-flight if this is a training sample
                if self.augment:
                    mel = self._apply_specaugment(mel)
                return mel

        # 2) Compute from audio
        audio = self._load_audio(row)
        feats = self.fe(audio, sampling_rate=CFG["target_sr"],
                        return_tensors="np")
        mel   = feats.input_features[0].astype(np.float32).T  # (3000, 80)
        del audio

        # FIX 4 — apply SpecAugment in-flight if this is a training sample
        if self.augment:
            mel = self._apply_specaugment(mel)

        return mel

    # ── FIX 4 — SpecAugment ──────────────────────────────────────────────────

    def _apply_specaugment(self, mel: np.ndarray) -> np.ndarray:
        """
        Apply SpecAugment (Park et al., 2019) to a mel spectrogram.

        mel shape: (T=3000, F=80) — time-major, as stored in the cache.

        Two classes of masks are applied:
          • Time masks  : randomly zero `t` consecutive time frames, repeated
                          `specaugment_time_masks` times.
          • Frequency masks: randomly zero `f` consecutive mel bins, repeated
                             `specaugment_freq_masks` times.

        Mask widths are sampled uniformly from [0, max_width], matching the
        standard Whisper training protocol.
        """
        mel  = mel.copy()          # never mutate the cached array
        T, F = mel.shape           # (3000, 80)

        # Time masking
        for _ in range(CFG["specaugment_time_masks"]):
            t  = random.randint(0, CFG["specaugment_time_mask_width"])
            t0 = random.randint(0, max(T - t, 0))
            mel[t0 : t0 + t, :] = 0.0

        # Frequency masking
        for _ in range(CFG["specaugment_freq_masks"]):
            f  = random.randint(0, CFG["specaugment_freq_mask_width"])
            f0 = random.randint(0, max(F - f, 0))
            mel[:, f0 : f0 + f] = 0.0

        return mel

    # ── Audio ─────────────────────────────────────────────────────────────────

    def _load_audio(self, row: pd.Series) -> np.ndarray:
        path = self._remap_path(str(row.processed_path))

        if _HAS_SOUNDFILE:
            try:
                audio, sr = sf.read(path, dtype="float32")
                if audio.ndim > 1:
                    audio = audio.mean(axis=1)
                if sr != CFG["target_sr"]:
                    audio = librosa.resample(audio, orig_sr=sr,
                                             target_sr=CFG["target_sr"])
            except Exception:
                audio, _ = librosa.load(path, sr=CFG["target_sr"], mono=True)
        else:
            audio, _ = librosa.load(path, sr=CFG["target_sr"], mono=True)

        has_ts  = str(row.get("has_timestamps", "True")).lower() == "true"
        max_len = self.max_audio_len
        if not has_ts and len(audio) > max_len:
            start = random.randint(0, len(audio) - max_len)
            audio = audio[start : start + max_len]
        audio = audio[:max_len]
        if len(audio) < max_len:
            audio = np.pad(audio, (0, max_len - len(audio)))
        return audio.astype(np.float32)

    # ── Tokens ────────────────────────────────────────────────────────────────

    def _get_tokens(
        self, idx: int, row: pd.Series
    ) -> Tuple[np.ndarray, np.ndarray]:
        if idx in self._tok_cache:
            return self._tok_cache[idx]

        text_ids = self.tok.encode(str(row.transcript),
                                   add_special_tokens=False)
        full_seq = self._prefix + text_ids + [self._eot]
        if len(full_seq) > self.max_label_len + 1:
            full_seq = full_seq[: self.max_label_len + 1]

        dec_in = full_seq[:-1]
        labels = full_seq[1:]
        pad    = self.max_label_len - len(dec_in)
        if pad > 0:
            dec_in = dec_in + [self._pad]  * pad
            labels = labels + [-100] * pad

        result = (
            np.array(dec_in, dtype=np.int32),
            np.array(labels, dtype=np.int32),
        )
        self._tok_cache[idx] = result
        return result


logger.info("EITDataset class defined (SpecAugment enabled).")

## Cell 6 — Data helpers  (leakage check, sample weights, dev prep)

In [ ]:
def assert_no_leakage(train_df: pd.DataFrame,
                      dev_df: pd.DataFrame,
                      test_df: pd.DataFrame) -> None:
    train_sp = set(train_df.speaker_id)
    dev_sp   = set(dev_df.speaker_id)
    test_sp  = set(test_df.speaker_id)
    td = train_sp & dev_sp
    tt = train_sp & test_sp
    assert not td, f"LEAKAGE train<->dev: {sorted(td)[:5]}"
    assert not tt, f"LEAKAGE train<->test: {sorted(tt)[:5]}"
    logger.info("No speaker leakage  (train/dev=empty, train/test=empty)")


def build_sample_weights(df: pd.DataFrame) -> np.ndarray:
    """Per-sample weights that balance datasets and L1 groups within INMIGRA."""
    weights = np.zeros(len(df), dtype=np.float64)
    for ds_name, ds_weight in CFG["dataset_weights"].items():
        mask  = (df.dataset == ds_name).values
        count = mask.sum()
        if count > 0:
            weights[mask] = ds_weight / count

    inmigra = (df.dataset == "Nebrija-INMIGRA").values
    if inmigra.sum() > 0 and "l1_group" in df.columns:
        sub    = df[inmigra]
        l1_cnt = sub.l1_group.value_counts()
        l1_w   = sub.l1_group.map(
            lambda l: 1.0 / l1_cnt.get(l, 1)
        ).values.astype(np.float64)
        l1_w   = l1_w / l1_w.sum() * inmigra.sum()
        weights[inmigra] *= l1_w

    total = weights.sum()
    return weights / total if total > 0 else np.full(len(df), 1.0 / len(df))


def prepare_dev_df(dev_csv: str, manifest_csv: str) -> pd.DataFrame:
    dev      = pd.read_csv(dev_csv)
    manifest = pd.read_csv(manifest_csv)
    cols     = ["utterance_id", "processed_path", "snr_db",
                "processed_duration_sec", "was_chunked",
                "rejected", "rejection_reason"]
    avail    = [c for c in cols if c in manifest.columns]
    dev      = dev.merge(manifest[avail], on="utterance_id", how="left")
    dev      = dev[dev.processed_path.notna()].copy()
    if "rejected" in dev.columns:
        dev  = dev[dev.rejected != True].copy()
    return dev.reset_index(drop=True)


logger.info("Data helper functions defined.")

## Cell 7 — Load data  &  build datasets

In [ ]:
logger.info("Loading CSVs ...")
train_df = pd.read_csv(CFG["train_csv"])
if "rejected" in train_df.columns:
    train_df = train_df[train_df.rejected != True].copy()
train_df = train_df[train_df.processed_path.notna()].reset_index(drop=True)

dev_df   = prepare_dev_df(CFG["dev_csv"], CFG["manifest_csv"])
test_df  = pd.read_csv(CFG["test_csv"])

logger.info("  train : %d samples", len(train_df))
logger.info("  dev   : %d samples", len(dev_df))
logger.info("  test  : %d samples", len(test_df))

_bad = train_df[
    ~train_df.processed_path.str.startswith("data/") &
    ~train_df.processed_path.str.startswith("/")
]
if len(_bad):
    logger.warning("%d rows have unexpected processed_path format", len(_bad))
else:
    logger.info("  all processed_path values match expected 'data/...' format")

assert_no_leakage(train_df, dev_df, test_df)

logger.info("Loading processor from %s ...", CFG["base_model"])
processor = WhisperProcessor.from_pretrained(
    CFG["base_model"], language=CFG["language"], task=CFG["task"]
)
processor.tokenizer.set_prefix_tokens(
    language=CFG["language"], task=CFG["task"]
)

# augment=True activates in-flight SpecAugment on training samples (FIX 4)
train_dataset = EITDataset(
    train_df, processor,
    mel_cache_dir = CFG["mel_cache_writable"],
    audio_root    = CFG["audio_root"],
    augment       = True,       # ← was True before but had no effect; now active
)
dev_dataset = EITDataset(
    dev_df, processor,
    mel_cache_dir = CFG["mel_cache_writable"],
    audio_root    = CFG["audio_root"],
    augment       = False,
)

sample_weights   = build_sample_weights(train_df)
weighted_sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(train_df),
    replacement = True,
)

logger.info("Datasets and sampler ready.")

## Cell 8 — Data collator

In [ ]:
@dataclass
class WhisperDataCollator:
    """
    Collates EITDataset samples into a batch.
    input_features  → (B, 80, 3000)
    decoder_input_ids / labels → padded to longest in batch
    """
    processor: WhisperProcessor
    max_label_len: int = 224

    def __call__(
        self, features: List[Dict[str, torch.Tensor]]
    ) -> Dict[str, torch.Tensor]:
        input_features = torch.stack(
            [f["input_features"] for f in features]
        )

        label_tensors  = [f["labels"]            for f in features]
        dec_in_tensors = [f["decoder_input_ids"] for f in features]

        labels        = torch.stack(label_tensors)
        decoder_input = torch.stack(dec_in_tensors)

        return {
            "input_features":    input_features,
            "decoder_input_ids": decoder_input,
            "labels":            labels,
        }


collator = WhisperDataCollator(processor, max_label_len=CFG["max_label_len"])
logger.info("WhisperDataCollator ready.")

## Cell 9 — Load model  &  apply LoRA

In [ ]:
logger.info("Loading base model: %s", CFG["base_model"])

# Load in float32; the HuggingFace Trainer handles fp16 autocast transparently
# via TrainingArguments(fp16=True), avoiding dtype mismatches with DataParallel.
base_model = WhisperForConditionalGeneration.from_pretrained(
    CFG["base_model"],
    torch_dtype = torch.float32,
)

# ── Generation config ────────────────────────────────────────────────────────
# Force Spanish transcription tokens.
# FIX 5 — add repetition_penalty and no_repeat_ngram_size to the generation
# config so every call to model.generate() is protected against hallucination
# loops by default (no need to pass them explicitly at inference time either).
decoder_prompt_ids = processor.get_decoder_prompt_ids(
    language=CFG["language"], task=CFG["task"]
)
base_model.generation_config.forced_decoder_ids  = decoder_prompt_ids
base_model.generation_config.suppress_tokens     = []
base_model.generation_config.repetition_penalty  = CFG["repetition_penalty"]
base_model.generation_config.no_repeat_ngram_size = CFG["no_repeat_ngram_size"]

# ── Apply LoRA ───────────────────────────────────────────────────────────────
# FIX 2 — correct target_modules.
#
# Previous version:
#   ["q_proj", "v_proj", "k_proj", "out_proj",
#    "encoder.layers.*.self_attn.q_proj",  # redundant wildcard
#    "encoder.layers.*.self_attn.v_proj"]  # redundant wildcard
#   → This created duplicate/undefined PEFT targets AND silently excluded
#     decoder cross-attention, which is exactly where the model learns to map
#     L2 acoustic patterns to correct Spanish tokens.
#
# Fixed version:
#   ["q_proj", "v_proj", "k_proj", "out_proj"]
#   → PEFT matches these short names against every Linear module in the
#     model tree, covering encoder self-attn, decoder self-attn, AND
#     decoder cross-attn uniformly.  No duplicates, no missed layers.
lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = CFG["lora_r"],
    lora_alpha     = CFG["lora_alpha"],
    lora_dropout   = CFG["lora_dropout"],
    target_modules = CFG["target_modules"],
    bias           = "none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model = model.to(DEVICE)
logger.info("Model on device: %s  (dtype=%s)",
            next(model.parameters()).device,
            next(model.parameters()).dtype)

## Cell 10 — compute_metrics callback  (WER / CER for Trainer)

In [ ]:
def make_compute_metrics(tokenizer, max_samples: int):
    """
    Returns the compute_metrics callable expected by Seq2SeqTrainer.

    Applies post_process_transcript to guard against hallucination loops
    before computing WER/CER.  This ensures the metric used for checkpoint
    selection reflects real transcription quality rather than being dominated
    by repeated-token outputs.
    """
    def compute_metrics(eval_pred) -> Dict[str, float]:
        preds, labels = eval_pred

        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        pred_strs  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        label_strs = tokenizer.batch_decode(labels, skip_special_tokens=True)

        n = min(len(pred_strs), max_samples)
        pred_strs  = pred_strs[:n]
        label_strs = label_strs[:n]

        refs = [normalize_text(r) for r in label_strs]
        # FIX 5 — apply hallucination post-processor before scoring
        hyps = [post_process_transcript(normalize_text(h)) for h in pred_strs]

        metrics = compute_metrics_raw(refs, hyps)

        logger.info(
            "  [eval]  WER=%.4f  CER=%.4f  90%%agree=%.4f  (n=%d)",
            metrics["wer"], metrics["cer"], metrics["90pct_agreement"], n
        )
        return metrics

    return compute_metrics


compute_metrics_fn = make_compute_metrics(
    processor.tokenizer,
    CFG["max_eval_gen_samples"],
)
logger.info("compute_metrics callback ready  (eval cap: %d samples)",
            CFG["max_eval_gen_samples"])

## Cell 11 — Custom Trainer  (preserves weighted sampler)

In [ ]:
class WeightedSeq2SeqTrainer(Seq2SeqTrainer):
    """
    Seq2SeqTrainer that uses our WeightedRandomSampler for training,
    preserving the Nebrija-INMIGRA / WOCAE / SPLLOC1 dataset balance.
    """

    def __init__(self, *args, train_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._train_sampler_override = train_sampler

    def _get_train_sampler(self, dataset=None):
        if self._train_sampler_override is not None:
            return self._train_sampler_override
        return super()._get_train_sampler(dataset)


class MetricsLoggerCallback(TrainerCallback):
    """Appends eval metrics to a JSON file after each eval step."""

    def __init__(self, output_path: str) -> None:
        self.output_path = Path(output_path)
        self.history: List[Dict] = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            entry = {"step": state.global_step, **metrics}
            self.history.append(entry)
            with open(self.output_path, "w") as f:
                json.dump(self.history, f, indent=2, default=str)


logger.info("WeightedSeq2SeqTrainer and callbacks defined.")

## Cell 12 — TrainingArguments  &  build Trainer

In [ ]:
# FIX 3 — epoch-based training with cosine LR decay.
#
# The previous approach used max_steps=1500, which covered <1 full epoch on
# the 56 K-sample training set.  The early-stopping best checkpoint fell at
# step 600, meaning the model barely warmed up.  Epoch-based training with
# cosine decay lets the LR anneal naturally, avoids premature stopping, and
# ensures every sample is seen multiple times.
#
# Epoch estimate:
#   ~56,000 train rows  /  effective_batch 32  = ~1,750 steps per epoch
#   10 epochs  = ~17,500 total optimisation steps
#   warmup_ratio=0.05  → ~875 warmup steps
#
# eval_steps=500 gives ~3-4 evaluations per epoch — fine-grained enough to
# catch early divergence without excessive overhead on large-v2.

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CFG["output_dir"],
    per_device_train_batch_size = CFG["batch_size"],
    per_device_eval_batch_size  = CFG["batch_size"],
    gradient_accumulation_steps = CFG["grad_accum"],
    learning_rate               = CFG["learning_rate"],

    # FIX 3 — epoch-based schedule
    num_train_epochs            = CFG["num_train_epochs"],
    lr_scheduler_type           = CFG["lr_scheduler_type"],   # "cosine"
    warmup_ratio                = CFG["warmup_ratio"],         # 5% of total steps

    fp16                        = CFG["fp16"],
    weight_decay                = CFG["weight_decay"],

    # Cuts ~40% activation memory at ~20% slower backward — essential on T4
    gradient_checkpointing      = True,

    # Evaluation — steps-based so we get ~3-4 checkpoints per epoch regardless
    # of epoch length.  Increase eval_steps for faster iteration if needed.
    eval_strategy               = "steps",
    eval_steps                  = CFG["eval_steps"],
    predict_with_generate       = True,
    generation_max_length       = CFG["generation_max_length"],

    # Checkpointing
    save_strategy               = "steps",
    save_steps                  = CFG["save_steps"],
    save_total_limit            = 2,
    save_only_model             = True,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,

    # Logging
    logging_dir                 = CFG["log_dir"],
    logging_steps               = CFG["logging_steps"],
    report_to                   = ["tensorboard"],

    dataloader_num_workers      = CFG["dataloader_workers"],
    dataloader_pin_memory       = True,
    seed                        = SEED,
)

metrics_logger = MetricsLoggerCallback(
    output_path=Path(CFG["results_dir"]) / "eval_history.json"
)

trainer = WeightedSeq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = dev_dataset,
    data_collator    = collator,
    compute_metrics  = compute_metrics_fn,
    processing_class = processor.feature_extractor,
    train_sampler    = weighted_sampler,
    callbacks        = [
        EarlyStoppingCallback(
            early_stopping_patience = CFG["early_stopping_patience"]
        ),
        metrics_logger,
    ],
)

logger.info("Trainer ready.")
logger.info(
    "Training plan: %d epochs  |  eff. batch %d  |  ~%d steps/epoch  |  LR %g (%s)",
    CFG["num_train_epochs"],
    CFG["batch_size"] * CFG["grad_accum"],
    math.ceil(len(train_df) / (CFG["batch_size"] * CFG["grad_accum"])),
    CFG["learning_rate"],
    CFG["lr_scheduler_type"],
)

## Cell 13 — Train

In [ ]:
logger.info("Checking mel cache ...")
n_cached = sum(1 for _ in Path(CFG["mel_cache_writable"]).glob("*.npy"))
n_total  = len(train_df) + len(dev_df)
if n_cached < n_total:
    logger.info(
        "  %d / %d mels cached — first epoch will compute the rest.",
        n_cached, n_total
    )
else:
    logger.info("  mel cache fully warm (%d files)", n_cached)

logger.info("=" * 60)
logger.info("Starting training ...")
t0 = time.perf_counter()

train_result = trainer.train()

elapsed = time.perf_counter() - t0
logger.info("Training complete in %.0f min", elapsed / 60)
logger.info("=" * 60)

trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

## Cell 14 — Post-training evaluation  (full dev set, WER / CER)

In [ ]:
logger.info("Running post-training evaluation on full dev set ...")

model.eval()
_model_dtype = next(model.parameters()).dtype

# FIX 5 — build decoder_prompt_ids once; generation config already has
# repetition_penalty and no_repeat_ngram_size baked in from Cell 9.
_forced_ids = processor.get_decoder_prompt_ids(
    language=CFG["language"], task=CFG["task"]
)

references, hypotheses = [], []
for idx in range(len(dev_dataset)):
    sample = dev_dataset[idx]
    mel_in = sample["input_features"].unsqueeze(0).to(DEVICE, dtype=_model_dtype)

    with torch.no_grad():
        pred_ids = model.generate(
            input_features     = mel_in,
            max_new_tokens     = CFG["generation_max_length"],
            forced_decoder_ids = _forced_ids,
            # repetition_penalty and no_repeat_ngram_size are already in
            # model.generation_config from Cell 9 — no need to repeat here.
        )

    raw_hyp = processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True)
    raw_ref = str(dev_df.iloc[idx].transcript)

    # FIX 5 — apply post-processing before storing
    hypotheses.append(post_process_transcript(normalize_text(raw_hyp)))
    references.append(normalize_text(raw_ref))

    if idx % 100 == 0:
        logger.info("  Decoded %d / %d ...", idx, len(dev_dataset))

final_metrics = compute_metrics_raw(references, hypotheses)
logger.info("=" * 60)
logger.info("Post-training dev results:")
logger.info("  WER        = %.4f", final_metrics["wer"])
logger.info("  CER        = %.4f", final_metrics["cer"])
logger.info("  90%%agree  = %.4f", final_metrics["90pct_agreement"])
logger.info("=" * 60)

summary = {
    "dev_wer":             final_metrics["wer"],
    "dev_cer":             final_metrics["cer"],
    "dev_90pct_agreement": final_metrics["90pct_agreement"],
    "total_steps":         trainer.state.global_step,
    "training_time_sec":   elapsed,
    "best_checkpoint":     trainer.state.best_model_checkpoint,
    "train_loss":          train_result.metrics.get("train_loss"),
    "model":               CFG["base_model"],
    "lora_r":              CFG["lora_r"],
    "lora_targets":        CFG["target_modules"],
    "num_train_epochs":    CFG["num_train_epochs"],
}
summary_path = Path(CFG["results_dir"]) / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
logger.info("Summary saved -> %s", summary_path)

print("\n-- 5 random predictions --")
for i in random.sample(range(len(references)), min(5, len(references))):
    print(f"REF : {references[i]}")
    print(f"HYP : {hypotheses[i]}")
    print()

## Cell 15 — Fuse LoRA adapters into base weights  &  export

In [ ]:
logger.info("Fusing LoRA adapters into base model ...")

fused_model = model.merge_and_unload()
fused_model = fused_model.to(torch.float16)

fused_dir = Path(CFG["output_dir"]) / "fused"
fused_dir.mkdir(parents=True, exist_ok=True)

fused_model.save_pretrained(str(fused_dir))
processor.save_pretrained(str(fused_dir))

logger.info("Fused model + processor saved -> %s", fused_dir)

readme = f"""# Whisper-large-v2 EIT LoRA (fused)

Base: `{CFG['base_model']}`  |  Language: `{CFG['language']}`  |  Task: `{CFG['task']}`

## Improvements over whisper-small baseline
- **Base model**: large-v2 (1.55 B) vs small (74 M) — 20x capacity
- **LoRA targets**: all attention projections incl. decoder cross-attention
- **Training**: {CFG['num_train_epochs']} epochs with cosine LR decay
- **Augmentation**: in-flight SpecAugment (time + frequency masking)
- **Hallucination guard**: repetition_penalty={CFG['repetition_penalty']}, no_repeat_ngram_size={CFG['no_repeat_ngram_size']}

## Quick inference (HuggingFace)

```python
from transformers import pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="{fused_dir}",
    chunk_length_s=30,
)
result = pipe("audio.wav", generate_kwargs={{"language": "es", "task": "transcribe"}})
print(result["text"])
```

## Convert to MLX for local Mac inference

```bash
pip install mlx-whisper
python -m mlx_whisper.convert \\
    --model {fused_dir} \\
    --output-dir ./mlx-whisper-eit \\
    --dtype float16
```

## Dev metrics
WER: {final_metrics['wer']:.4f}  |  CER: {final_metrics['cer']:.4f}
90%-agreement: {final_metrics['90pct_agreement']:.4f}
Steps trained: {trainer.state.global_step}
"""
(fused_dir / "README.md").write_text(readme)
logger.info("README written.")

## Cell 16 — Zip  &  save output  (Kaggle Output panel)

In [ ]:
import shutil, os

best_ckpt = trainer.state.best_model_checkpoint
ckpt_root = Path(CFG["output_dir"])
for d in ckpt_root.iterdir():
    if d.is_dir() and d.name.startswith("checkpoint-"):
        if best_ckpt and str(d) == best_ckpt:
            logger.info("Keeping best checkpoint: %s", d)
        else:
            shutil.rmtree(d)
            logger.info("Deleted checkpoint to free space: %s", d)

zip_path = "/kaggle/working/whisper_eit_lora.zip"
logger.info("Zipping fused model to %s ...", zip_path)
shutil.make_archive(
    base_name = zip_path.replace(".zip", ""),
    format    = "zip",
    root_dir  = str(fused_dir.parent),
    base_dir  = fused_dir.name,
)
size_mb = Path(zip_path).stat().st_size / 1e6
logger.info("Done.  Archive size: %.1f MB", size_mb)
logger.info("Download it from the Kaggle Output panel on the right.")

shutil.make_archive(
    base_name = "/kaggle/working/results",
    format    = "zip",
    root_dir  = CFG["results_dir"],
)
logger.info("Results zipped to /kaggle/working/results.zip")

print("All done!")
print(f"  Model : {CFG['base_model']} + LoRA r={CFG['lora_r']}")
print(f"  WER   = {final_metrics['wer']:.4f}")
print(f"  CER   = {final_metrics['cer']:.4f}")
print(f"  90%%agree = {final_metrics['90pct_agreement']:.4f}")
print(f"  Steps trained = {trainer.state.global_step}")

---
## Appendix — Useful diagnostics (run on-demand)

### A) Check GPU memory usage

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    total_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Total     : {total_gb:.2f} GB")
    print(f"  Free est. : {total_gb - reserved:.2f} GB")

### B) Plot WER / CER curve

In [ ]:
history_path = Path(CFG["results_dir"]) / "eval_history.json"
if history_path.exists():
    import matplotlib.pyplot as plt
    with open(history_path) as f:
        hist = json.load(f)
    steps = [h["step"]                           for h in hist]
    wers  = [h.get("eval_wer", h.get("wer"))     for h in hist]
    cers  = [h.get("eval_cer", h.get("cer"))     for h in hist]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, wers, marker="o", label="WER", color="#e05c5c")
    ax.plot(steps, cers, marker="s", label="CER", color="#5c8ae0")
    ax.set_xlabel("Step")
    ax.set_ylabel("Error Rate")
    ax.set_title("WER / CER on dev set during training")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(CFG["results_dir"]) / "wer_curve.png", dpi=150)
    plt.show()
else:
    print("eval_history.json not found yet — run training first.")

### C) Resume from a checkpoint

In [ ]:
# Set RESUME_FROM to the checkpoint folder to continue interrupted training.
# Leave as None to start fresh.
RESUME_FROM = None   # e.g. "/kaggle/working/checkpoints/checkpoint-3500"

if RESUME_FROM:
    logger.info("Resuming from %s", RESUME_FROM)
    train_result = trainer.train(resume_from_checkpoint=RESUME_FROM)